This notebook presents a clean and practical implementation of a Retrieval-Augmented Generation (RAG) pipeline, covering text-to-document conversion, multi-model embeddings using OpenAI, Gemini, and Hugging Face, and efficient retrieval with a FAISS vector store.

It also explores distance_strategy options like EUCLIDEAN and COSINE, highlighting how the choice of similarity metric directly impacts retrieval accuracy—making the pipeline more precise, flexible, and aligned with real-world AI applications.

In [ ]:
!pip install langchain
!pip install langchain_google_genai
!pip install langchain_community
!pip install langchain_text_splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
import os

from google.colab import userdata
gem = userdata.get('Gemini')

# For OpenAI
# op = userdata.get('open1')

In [ ]:
os.environ["GOOGLE_API_KEY"]=gem

# For Openai : os.environ["OPENAI_API_KEY"]=op

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# For OpenAI
# from langchain_openai.embeddings import OpenAIEmbeddings

In [ ]:
emb_llm = GoogleGenerativeAIEmbeddings(
    model = "gemini-embedding-001"
)

# For OpenAI
# emb_llm = OpenAIEmbeddings(model = "text-embedding-3-small")
#By default, the length of the embedding vector is 1536 for text-embedding-3-small

In [ ]:
v1 = emb_llm.embed_query("Hi how are you")

In [ ]:
len(v1) # In Case of OpenAI we will get 1536

3072

In [ ]:
corpus = ["Hi how are you","apple is good","apple is not good"]

In [ ]:
v2 = emb_llm.embed_documents(corpus)

In [ ]:
len(v2[0])

3072

In [ ]:
len(v2[1])

3072

In [ ]:
# How to convert text into document
text = "Hi how are you"

In [ ]:
from langchain_core.documents import Document

# This is use to convert text into the document

In [ ]:
Document(page_content=text)

Document(metadata={}, page_content='Hi how are you')

In [ ]:
doc = [Document(page_content=text) for text in corpus]

In [ ]:
doc

[Document(metadata={}, page_content='Hi how are you'),
 Document(metadata={}, page_content='apple is good'),
 Document(metadata={}, page_content='apple is not good')]

In [ ]:
emb_llm.embed_documents(doc[0].page_content)

[[-0.016372832,
  0.013813561,
  0.013894072,
  -0.0782227,
  -0.016500361,
  0.013281048,
  0.016156241,
  0.006484267,
  0.0027471962,
  -0.0077986256,
  -0.009503913,
  -0.016797954,
  0.004657148,
  -0.0048378464,
  0.17070496,
  -0.0029155298,
  0.009090932,
  0.011881032,
  0.00011048462,
  -0.0018181393,
  0.006870429,
  -0.0029864602,
  0.015406045,
  -0.021015288,
  0.019104788,
  -0.021567952,
  0.0055746743,
  0.017206125,
  0.030846562,
  -0.00022085459,
  -0.0047296276,
  0.015639424,
  0.0024307945,
  0.01653014,
  0.021424696,
  0.018148843,
  0.017227367,
  -0.0126292035,
  0.01957177,
  0.023071324,
  0.009647776,
  0.003146558,
  0.0032347487,
  -0.012575404,
  -0.00023258911,
  0.011938705,
  -0.003232339,
  -0.012547341,
  0.0036637946,
  0.024993334,
  0.00066301675,
  0.0033077027,
  -0.01866358,
  -0.26272243,
  -0.008697337,
  0.0048631146,
  -0.007320559,
  0.017455146,
  0.006119266,
  -0.016306447,
  -0.006843881,
  0.012445126,
  0.0013659864,
  -0.023019735

In [ ]:
[emb_llm.embed_documents(docs.page_content) for docs in doc]

[[[-0.016372832,
   0.013813561,
   0.013894072,
   -0.0782227,
   -0.016500361,
   0.013281048,
   0.016156241,
   0.006484267,
   0.0027471962,
   -0.0077986256,
   -0.009503913,
   -0.016797954,
   0.004657148,
   -0.0048378464,
   0.17070496,
   -0.0029155298,
   0.009090932,
   0.011881032,
   0.00011048462,
   -0.0018181393,
   0.006870429,
   -0.0029864602,
   0.015406045,
   -0.021015288,
   0.019104788,
   -0.021567952,
   0.0055746743,
   0.017206125,
   0.030846562,
   -0.00022085459,
   -0.0047296276,
   0.015639424,
   0.0024307945,
   0.01653014,
   0.021424696,
   0.018148843,
   0.017227367,
   -0.0126292035,
   0.01957177,
   0.023071324,
   0.009647776,
   0.003146558,
   0.0032347487,
   -0.012575404,
   -0.00023258911,
   0.011938705,
   -0.003232339,
   -0.012547341,
   0.0036637946,
   0.024993334,
   0.00066301675,
   0.0033077027,
   -0.01866358,
   -0.26272243,
   -0.008697337,
   0.0048631146,
   -0.007320559,
   0.017455146,
   0.006119266,
   -0.016306447,
 

In [ ]:
# Use a vectore store -- Faiss

from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import DistanceStrategy

In the FAISS we want to give 3 things
(Document,Embeddings,Distance)
Distance = To perform Similarity search

Note: Inside vector database and vector store It will be having different types of retrievers

And this retrievers(tools) by using which it can get similar vector based on the query and these retriever are actually using different types of distance to calculate which vector are closer

#### IMP NOTE: when we use FAISS we no need to use embedding step

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 48.6 MB/s eta 0:00:00


In [ ]:
# Step 1 = To create a vector store

corpus = ["Hi how are you","apple is good","apple is not good"]
emb_llm = GoogleGenerativeAIEmbeddings(
    model = "gemini-embedding-001"
)

In [ ]:
vs = FAISS.from_texts(texts=corpus,embedding=emb_llm, distance_strategy=DistanceStrategy.COSINE)

Intrenally it will take all this corpus , loop through this corpus, every document give it to embedding, and convert it into a vector, that vector will be taken by faiss and store inside RAM and this is going to give me object which i am calling as a vector store

In [ ]:
vs.save_local("vs1")

In [ ]:
vs1 = vs.load_local("/content/vs1",embeddings=emb_llm, allow_dangerous_deserialization=True)

In [ ]:
vs.similarity_search("Hi how are you",k=2) #it return 2 closed vector k = 2 and it is a retriever

[Document(id='baf01823-fb6e-4ac0-bad4-ca438c97d2d6', metadata={}, page_content='Hi how are you'),
 Document(id='3c5f7613-a48c-48df-82d8-e3f667feba6f', metadata={}, page_content='apple is good')]

In [ ]:
vs.similarity_search_with_score("Hi how are you",k=2) #it return 2 closed vector k = 2 and it is a retriever

[(Document(id='baf01823-fb6e-4ac0-bad4-ca438c97d2d6', metadata={}, page_content='Hi how are you'),
  np.float32(0.3710578)),
 (Document(id='3c5f7613-a48c-48df-82d8-e3f667feba6f', metadata={}, page_content='apple is good'),
  np.float32(0.81954384))]

similarity_search_with_score return = float32(0.81954384)) it is a euclidean distance and distance range is 0 to infinity

In [ ]:
vs22 = FAISS.from_texts(texts=corpus,embedding=emb_llm, distance_strategy=DistanceStrategy.EUCLIDEAN)

In [ ]:
vs22.save_local("vs2")

In [ ]:
vs2 = vs22.load_local("/content/vs2",embeddings=emb_llm, allow_dangerous_deserialization=True)

In [ ]:
vs2.similarity_search_with_score("Hi how are you",k=2) #it return 2 closed vector k = 2 and it is a retriever

[(Document(id='d570ab25-a6fb-4d7f-b4cb-7ff8caf80566', metadata={}, page_content='Hi how are you'),
  np.float32(0.3710578)),
 (Document(id='d19af345-c1d4-449b-9acf-3e7fd5c90499', metadata={}, page_content='apple is good'),
  np.float32(0.81954384))]

In [ ]:
vs2.similarity_search_with_score("Hi how are you",k=2) #it return 2 closed vector k = 2 and it is a retriever

[(Document(id='d570ab25-a6fb-4d7f-b4cb-7ff8caf80566', metadata={}, page_content='Hi how are you'),
  np.float32(0.3710578)),
 (Document(id='d19af345-c1d4-449b-9acf-3e7fd5c90499', metadata={}, page_content='apple is good'),
  np.float32(0.81954384))]

In [ ]:
!pip install transformers
!pip install sentence_transformers #Many of the huggingface embedding models are present into this library

In [ ]:
!pip install langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

Note: HuggingFaceEmbeddings it is a class provided by langchain to access any kind of embedding models present inside sentence-transformers library

In [ ]:
emb = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Note : HuggingFaceEmbeddings(model="all-MiniLM-L6-v2") : This will first access the huggingface and download the entire embeddings model locally and it will use your local CPU to perform embeddings


- It downloads the entire transformers models so now we no need to worry about the APIs.
- we transform embeddings n no of times for the hug data , But in OpenAI once we closed to the limit then we need to pay more.
